# Holiday Identification by Daily Profile Clustering

This notebook flags atypical days in one demand series by comparing each daily profile against its `(segment, weekday)` reference group.

It loads observed data, builds day-level profiles, measures distance to the group centroid, and compares detected outliers against the local holiday catalog.

This is exploratory notebook code, not part of the production pipeline.

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import analog_holidays.shared.identify_holidays as identify_holidays_module
identify_holidays_module = importlib.reload(identify_holidays_module)

from analog_holidays.shared.dataset_config import ACTIVE_CONFIG, list_dataset_regions
from analog_holidays.shared.identify_holidays import (
    build_holiday_groups,
    build_holiday_selector_features,
    build_wide_df,
    compare_outliers_holidays,
    compute_distances,
    detect_outliers,
    display_cluster_holiday_crosstab,
    display_cluster_38h_crosstab,
    find_holidays_not_detected,
    get_date_sets,
    get_hour_cols,
    load_holidays_catalog,
    load_results_data,
    plot_distance_distribution,
    plot_profiles_by_holiday,
    plot_profiles_by_segment_dow,
    print_summary,
    report_nth_monday_holidays,
    run_analog_cluster_38h_analysis,
    run_cluster_ab_validation,
    run_cluster_38h_analysis,
    run_cluster_atypical_analysis,
    run_holiday_ab_validation,
 )

DEMAND_PATH = ACTIVE_CONFIG.demand_path
HOLIDAYS_PATH = ACTIVE_CONFIG.notebook_holidays_path

print(f'Active dataset: {ACTIVE_CONFIG.key}')
print(f'Demand CSV: {DEMAND_PATH}')
print(f'Holidays: {HOLIDAYS_PATH}')


Active dataset: mx
Demand CSV: /home/uriel/GIT/analog_holidays/holidays/holiday_demand_mx.csv
Holidays: /home/uriel/GIT/analog_holidays/holidays/holidays_recognized.json


## 1. Configuration

In [ ]:
UNIQUE_ID = 'ERCOT_demand_ERCOT'

DATE_END = None
SELECTOR_FUTURE_END = '2026-12-31'
CLUSTERING_CRITERIUM = 'holiday_identity'
CLUSTERING_CRITERIA_CATALOG = identify_holidays_module.ANALOG_CLUSTER_CRITERIA_CATALOG.copy()
# Public criterion catalog exposed by the selector pipeline:
# - 'shape_pearson_CDE_map_FGH' keeps the current 38h shape-based C/D/E -> F/G/H mapping.
# - 'seasonal_heat_cold' groups holiday dates into heat vs cold seasons, mapping Spring/Summer to heat and Autumn/Winter to cold.
# - 'seasonal_winter_sprint_fall' groups holiday dates by year season and maps Winter/Spring/Fall/Summer to stable analog labels.
# - 'best_matching_weekday' groups holiday dates by the closest weekday-profile label assigned to each date.
# - 'observance_tier' groups holiday dates by how inhábil they actually are (working/partial/full), derived from observed_strength.
# - 'holiday_identity' gives every distinct holiday (by anchor name) its OWN analog cluster — the pure Similar-Days
#   hard filter: the downstream selector then only matches a target against other instances of the SAME holiday.


AVAILABLE_UNIQUE_IDS = list_dataset_regions()
if not AVAILABLE_UNIQUE_IDS:
    raise ValueError(f'No configured series were found in {DEMAND_PATH}.')
if UNIQUE_ID is None:
    UNIQUE_ID = AVAILABLE_UNIQUE_IDS[0]
elif UNIQUE_ID not in AVAILABLE_UNIQUE_IDS:
    raise ValueError(
        f'Series {UNIQUE_ID!r} not found. Available: {AVAILABLE_UNIQUE_IDS}'
    )

MONTH_NAMES = [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December',
]

SEGMENTS = [
    {'label': 'Spring', 'months': [3, 4, 5]},
    {'label': 'Summer', 'months': [6, 7, 8]},
    {'label': 'Autumn', 'months': [9, 10, 11]},
    {'label': 'Winter', 'months': [12, 1, 2]},
]

SEGMENTS = [{'label': month_name, 'months': [month_number]} for month_number, month_name in enumerate(MONTH_NAMES, start=1)]

OUTLIER_PERCENTILE = 95
EXCLUDE_KNOWN_HOLIDAYS_FROM_BASELINE = True
KNOWN_HOLIDAY_MIN_GROUP_PERCENTILE = 75
DISTANCE_METRIC = 'PEARSON'

N_CLUSTERS_PHOL    = 3   # clusters for 38-h eve+holiday profiles (section 8c-bis)
PREVIOUSLY_W_HOURS = 14  # hours taken from the eve day

WEEKDAY_NAMES = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

print(f'Series:   {UNIQUE_ID}')
print(f'DATE_END: {DATE_END}')
print(f'SELECTOR_FUTURE_END: {SELECTOR_FUTURE_END}')
print(f'CLUSTERING_CRITERIUM: {CLUSTERING_CRITERIUM}')
print(f'Criterion catalog: {tuple(CLUSTERING_CRITERIA_CATALOG)}')
print(f'Available series: {AVAILABLE_UNIQUE_IDS}')
display(
    pd.DataFrame(
        [
            {'criterion': criterion_name, 'description': criterion_description}
            for criterion_name, criterion_description in CLUSTERING_CRITERIA_CATALOG.items()
        ]
    )
)


## 2. Load observed data from CSV

In [ ]:
MONTH_TO_SEGMENT = {month: segment['label'] for segment in SEGMENTS for month in segment['months']}
SEGMENT_LABELS = [segment['label'] for segment in SEGMENTS]

df_raw = load_results_data(DEMAND_PATH, UNIQUE_ID)
if DATE_END is not None:
    cutoff_ts = pd.Timestamp(DATE_END)
    df_raw = df_raw[df_raw['ds'] < cutoff_ts].copy()
    if df_raw.empty:
        raise ValueError(f'No data available for {UNIQUE_ID} before {cutoff_ts.date()}.')

print(f'Series: {UNIQUE_ID}')
print(f'Records: {len(df_raw):,}')
if DATE_END is not None:
    print(f'DATE_END cutoff: {pd.Timestamp(DATE_END).date()} (exclusive)')
print(f'Range: {df_raw["ds"].min()} → {df_raw["ds"].max()}')
df_raw.head()

## 3. Pivot to wide format (1 row = 1 day, 24 columns = hours)

In [ ]:
df_wide = build_wide_df(df_raw, MONTH_TO_SEGMENT, exclude_years=[])
HOUR_COLS = get_hour_cols(df_wide)
year_min = int(df_wide.index.year.min())
year_max = int(df_wide.index.year.max())
df_holidays = load_holidays_catalog(HOLIDAYS_PATH, year_min, year_max)
df_holidays_display = df_holidays.copy()
KNOWN_HOLIDAY_DATES = (
    set(pd.to_datetime(df_holidays['date']).dt.normalize())
    if EXCLUDE_KNOWN_HOLIDAYS_FROM_BASELINE else set()
 )

print(f'Complete days: {len(df_wide)}')
print(f'Segments ({len(SEGMENT_LABELS)}):')
for segment_label in SEGMENT_LABELS:
    n_days = (df_wide['segment'] == segment_label).sum()
    print(f'  {segment_label}: {n_days} days')
print(f'Catalog holidays in range: {len(df_holidays)}')
print(f'Dates excluded from the baseline: {len(KNOWN_HOLIDAY_DATES)}')
df_wide.head()

## 4. Compute distances by segment and weekday

Distances are computed inside each `(segment, dow)` group against that group's reference centroid.

`DISTANCE_METRIC` controls whether the score emphasizes magnitude, shape, or both.

In [ ]:
df_dist = compute_distances(
    df_wide,
    HOUR_COLS,
    SEGMENT_LABELS,
    WEEKDAY_NAMES,
    distance_metric=DISTANCE_METRIC,
    reference_exclude_dates=KNOWN_HOLIDAY_DATES,
 )

print(f'Distance metric: {DISTANCE_METRIC}')
if DISTANCE_METRIC == 'PEARSON_EUCLIDIAN':
    print('  Euclidean and Pearson components combined after group-wise normalization')
elif DISTANCE_METRIC == 'EUCLIDIAN':
    print('  Euclidean distance only')
elif DISTANCE_METRIC == 'PEARSON':
    print('  1 - Pearson r only')
print(f'\nSegments: {SEGMENT_LABELS}')
print(f'Total records (day × group): {len(df_dist)}')
df_dist.head(10)

## 6. Load the local recognized holidays catalog (`holidays/holidays_recognized.json`)

In [ ]:
df_dist, df_outliers = detect_outliers(
    df_dist,
    OUTLIER_PERCENTILE,
    threshold_reference_only=EXCLUDE_KNOWN_HOLIDAYS_FROM_BASELINE,
    promote_dates=KNOWN_HOLIDAY_DATES,
    promote_min_group_percentile=KNOWN_HOLIDAY_MIN_GROUP_PERCENTILE,
)

print(f'Detected outliers: {len(df_outliers)}')
print(f'Out of a total of {len(df_dist)} days')
print(f'Percentage: {100 * len(df_outliers) / len(df_dist):.1f}%')
print(f'Known holidays rescued by within-group percentile: {int(df_dist["is_promoted_outlier"].sum())}')
df_outliers.head(20)

## 6. Load the local recognized holidays catalog (`holidays/holidays_recognized.json`)

In [ ]:
print(f'Generated holidays: {len(df_holidays_display)} (from {year_min} to {year_max})')
df_holidays_display.tail(15)


## 6b. Holidays with an nth-Monday rule

Three civic holidays moved from fixed dates to observed Mondays after the 2006 Federal Labor Law reform. Years before 2006 keep the historical fixed date.

| Holiday | Before 2006 | Since 2006 |
|---|---|---|
| Constitution Day | February 5 | 1st Monday of February |
| Benito Juarez's Birthday | March 21 | 3rd Monday of March |
| Mexican Revolution Day | November 20 | 3rd Monday of November |

In [ ]:
df_nth = report_nth_monday_holidays(df_holidays)

print(f'Holidays with an nth-Monday rule (since 2006): {len(df_nth)} occurrences')
display(
    df_nth.rename(columns={
        'holiday_name': 'Holiday',
        'date': 'Observed date',
        'weekday_name': 'Weekday',
        'labor_law_rule': 'LFT rule',
    })
)

## 7. Compare outliers vs known holidays

In [ ]:
df_match, stats = compare_outliers_holidays(df_outliers, df_holidays)
df_outliers_cmp = df_match
df_match_display = df_match.copy()

n_total = stats['n_total']
n_match = stats['n_match']
n_unknown = stats['n_unknown']

print('Outlier comparison against the holiday catalog')
print(f'Total outliers: {n_total}')
print(f'Match a known holiday: {n_match} ({100 * n_match / n_total:.0f}%)')
print(f'No catalog match: {n_unknown} ({100 * n_unknown / n_total:.0f}%)')

cols_display = ['date', 'dow_name', 'segment', 'distance', 'holiday_name']

print('\nKnown holidays detected as outliers')
display(
    df_match_display[df_match_display['is_known_holiday']]
    .sort_values('holiday_name')[cols_display]
 )

print('\nOutliers without a catalog match')
display(
    df_match_display[~df_match_display['is_known_holiday']]
    .sort_values('holiday_name')[cols_display]
 )

In [ ]:
detected_dates = set(pd.to_datetime(df_outliers_cmp['date']).dt.normalize())
all_dates_in_data = set(df_wide.index.normalize())

df_missed = find_holidays_not_detected(
    df_holidays,
    all_dates_in_data,
    detected_dates,
    WEEKDAY_NAMES,
 )
df_missed_display = df_missed.copy()

print(f'Known holidays present in the data but not detected as outliers: {len(df_missed)}')
if not df_missed.empty:
    display(df_missed_display.sort_values('holiday_name')[['holiday_name', 'dow_name', 'date']])

date_sets = get_date_sets(df_outliers_cmp, df_holidays, all_dates_in_data)
outlier_dates_set = date_sets['outlier_dates_set']
holiday_dates_set = date_sets['holiday_dates_set']
match_dates_set = date_sets['match_dates_set']
unknown_dates_set = date_sets['unknown_dates_set']
missed_dates_set = date_sets['missed_dates_set']

## 8. Visualization — profiles by weekday and outliers

In [ ]:
plot_profiles_by_segment_dow(
    df_wide, HOUR_COLS, SEGMENT_LABELS, WEEKDAY_NAMES,
    match_dates_set, unknown_dates_set, missed_dates_set, UNIQUE_ID,
    df_holidays=df_holidays_display,
)

Detected profiles use the same colors as the plotting helpers:

| Color | Meaning |
|---|---|
| Green | Detected outlier that matches a catalog holiday |
| Red | Detected outlier without a catalog match |
| Blue | Catalog holiday that was not flagged as an outlier |
| Black dashed line | Group centroid |
| Faint gray | Regular days |

## 8b. Hourly profiles by holiday

Each subplot overlays all available yearly profiles for one holiday present in the data.

| Color | Meaning |
|---|---|
| Green | Year detected as an outlier |
| Blue | Year present but not detected as an outlier |
| Black dashed line | Average holiday profile across years |

In [ ]:
holiday_groups = build_holiday_groups(df_holidays_display, df_wide.index)

plot_profiles_by_holiday(
    df_wide, holiday_groups, df_holidays_display, HOUR_COLS,
    match_dates_set, outlier_dates_set, UNIQUE_ID,
)

## 8c. Cluster atypical profiles

This view clusters the detected atypical profiles and compares each cluster centroid against weekday reference profiles.

In [ ]:
N_CLUSTERS = 2
CLUSTER_COLORS = [
    '#f58231',  '#4363d8', '#3cb44b', '#e6194b',
    '#911eb4', '#42d4f4', '#f032e6', '#bfef45',
]

cluster_results = run_cluster_atypical_analysis(
    df_wide,
    match_dates_set,
    set(),
    outlier_dates_set,
    N_CLUSTERS,
    HOUR_COLS,
    df_holidays_display,
    CLUSTER_COLORS,
    UNIQUE_ID,
)
df_atyp = cluster_results['df_atyp']
df_sim = cluster_results['df_sim']


## 8c-bis. Cluster: 38-h event profile (eve last 14 h + holiday 24 h)

For every confirmed holiday `D`, a single **38-hour feature vector** is built:

| Segment | Hours | Source |
|---------|-------|--------|
| Pre-holiday eve | last `PREVIOUSLY_W_HOURS = 14` h of day `D−1` (h10…h23) | `df_wide[D−1]` |
| Holiday | full 24 h of day `D` (h0…h23) | `df_wide[D]` |

KMeans is then applied to these 38-column profiles.  
Events where the eve day is missing from the data are skipped.


In [ ]:
cluster_38h_results = run_cluster_38h_analysis(
    df_wide=df_wide,
    match_dates_set=match_dates_set,
    df_holidays_display=df_holidays_display,
    hour_cols=HOUR_COLS,
    cluster_colors=CLUSTER_COLORS,
    unique_id=UNIQUE_ID,
    n_clusters=N_CLUSTERS_PHOL,
    previously_w_hours=PREVIOUSLY_W_HOURS,
)
df_phol      = cluster_38h_results['df_phol']
df_days_phol = cluster_38h_results['df_days_phol']

## 8d. Cluster DOW similarity

For each cluster, ranks all **7 day-of-week** reference profiles (Mon–Sun) by mean Pearson correlation, then tests statistically whether the top-ranked DOW is significantly better than the runner-up.

This avoids the ad-hoc Sat/Sun assumption: a cluster that resembles Wednesday and Sunday equally will show up as "not distinguishable", while a cluster that truly behaves like Saturday will have a clear green winner column.

| Output | Meaning |
|---|---|
| **Table 1 — mean corr** | Mean Pearson r of each cluster against each DOW centroid (same month). Green = highest, pink = lowest per row. |
| **Table 2 — votes** | Number of days in the cluster where each DOW was the best individual match. |
| **Table 3 — per-day** | Every atypical day with its 7 correlation values; green = best DOW for that day. |
| `best_dow` | DOW with highest mean r (winner). |
| `runner_up_dow` | Second-best DOW. |
| `gap_corr` | Mean r of winner minus mean r of runner-up. |
| `wilcoxon_pvalue` | One-sided Wilcoxon on (r_winner − r_runner_up) per day: p < alpha → winner is unambiguously better. |
| `cluster_type` | Winner DOW label if significant, `'unclear'` otherwise. |


In [ ]:
AB_ALPHA = 0.10

cluster_ab_results = run_cluster_ab_validation(
    df_wide=df_wide,
    df_atyp=df_atyp,
    outlier_dates_set=outlier_dates_set,
    hour_cols=HOUR_COLS,
    cluster_colors=CLUSTER_COLORS,
    alpha=AB_ALPHA,
    df_holidays=df_holidays_display,
)

# cluster_type: winner DOW label (e.g. 'Sat', 'Sun') if statistically clear, else 'unclear'
cluster_type_map = dict(
    zip(
        cluster_ab_results['summary_df']['cluster'],
        cluster_ab_results['summary_df']['cluster_type'],
    )
)
print('cluster_type_map =', cluster_type_map)


In [ ]:
df_cluster_holiday_crosstab = display_cluster_holiday_crosstab(
    df_atyp=df_atyp,
    df_holidays=df_holidays_display,
    cluster_colors=CLUSTER_COLORS,
)


## 9. Holiday Selector Feature Schema

The selector table `df_holiday_selector_features` combines calendar rules and profile-based labels so analog candidates can be filtered or ranked by context before distance ranking.

| Field | Meaning | Typical values / source |
|---|---|---|
| `unique_id` | Demand series identifier that produced the profile labels. This keeps the exported selector series-aware when multiple regions share the same CSV. | `SEN_demand_SIN`, `OCC_demand_BAJ` |
| `holiday_name` | Name assigned to the row itself. For `H1`/`H2`/`H3` rows this is the holiday on that date. For `H4` rows this is the synthetic recovery label. | `Labor Day`, `Good Friday`, `Post-holiday recovery` |
| `anchor_holiday_name` | Anchor holiday used to identify the event family behind the row. For `H1`/`H2`/`H3` it matches `holiday_name`. For `H4` it is the last holiday in the immediately preceding run. | `Christmas Day`, `New Year's Day` |
| `date` | Calendar date represented by the row. | `2020-05-01` |
| `holiday_day_type` | H-day taxonomy label. `H1` = eve day, `H2` = core or standalone holiday, `H3` = consecutive post-holiday day, `H4` = first recovery day after a run of length >= 2. | `H1`, `H2`, `H3`, `H4` |
| `weekday_name` | Literal weekday of `date`, independent of demand similarity. | `Monday`, `Saturday`, `Sunday` |
| `day_class_code` | Compact labor-calendar code used by the selector. `1` = weekday, `2` = Saturday, `3` = Sunday. | `1`, `2`, `3` |
| `day_class_name` | Expanded text version of `day_class_code`. | `Weekday`, `Saturday`, `Sunday` |
| `season` | Meteorological season derived from the month. | `Winter`, `Spring`, `Summer`, `Autumn` |
| `date_rule` | Calendar rule behind the date. `fixed_date` = fixed official date, `observed_monday_rule` = Monday-observed civic holiday after the 2006 labor reform, `movable_date` = Easter-based movable holiday, `derived_recovery_day` = synthetic `H4` row. | `fixed_date`, `observed_monday_rule`, `movable_date`, `derived_recovery_day` |
| `is_fixed_date` | Boolean helper flag for fixed-date observances. This is also `True` for pre-2006 years of Monday-observed civic holidays, when they still fell on their original fixed date. | `True`, `False` |
| `is_observed_monday_rule` | Boolean helper flag for civic holidays shifted to Monday by the 2006 labor reform. | `True`, `False` |
| `best_matching_weekday` | Best weekday match at the individual-day level, taken from section 8d (`best_dow`) and expanded to full weekday names. This is the closest weekday profile for that specific date. | `Saturday`, `Sunday`, `Wednesday` |
| `daily_profile_cluster` | DOW-agnostic letter for the daily-profile cluster from section 8c. It is derived from `daily_profile_cluster_id` in ascending numeric order, so the current notebook typically shows `A`, `B`, etc. | `A`, `B`, `C` |
| `daily_profile_cluster_id` | Raw numeric KMeans cluster id from section 8c (atypical daily profiles). | `0`, `1`, `2` |
| `daily_profile_archetype` | Cluster-level weekday archetype inferred from section 8d (`cluster_type`). This is the human-readable interpretation of the cluster, such as `Saturday-like`, `Sunday-like`, or `unclear`. | `Saturday-like`, `Sunday-like`, `unclear` |
| `event_profile_cluster` | DOW-agnostic letter for the 38-h event-profile cluster from section 8c-bis. The current notebook typically shows `C`, `D`, `E`, etc. | `C`, `D`, `E` |
| `event_profile_cluster_id` | Raw numeric KMeans cluster id from section 8c-bis (38-h eve + holiday profile). | `0`, `1`, `2` |
| `analog_cluster_criterion` | Public criterion selected in section 11 to derive `analog_cluster` in the exported selector. This records which grouping rule produced the stable analog labels. | `seasonal_heat_cold`, `best_matching_weekday` |

Notes:

- `best_matching_weekday` is a per-date label, while `daily_profile_archetype` is a cluster-level label.
- `daily_profile_cluster` / `daily_profile_cluster_id` come from the 24-h atypical-day analysis in section 8c.
- `event_profile_cluster` / `event_profile_cluster_id` come from the 38-h eve+holiday analysis in section 8c-bis.
- `analog_cluster_criterion` is constant within one selector export and tells downstream notebooks how `analog_cluster` was derived.
- `unique_id` is exported so the selector and priors can coexist for multiple series in the same CSV without mixing labels across regions.
- `H4` rows are derived automatically and may not have event-profile labels because the 38-h clustering is defined only for confirmed holiday dates.
- Missing values in cluster-related columns are acceptable when a row was not part of the corresponding upstream analysis.
- For future candidates, profile-based labels are inferred first within the same `anchor_holiday_name` + `holiday_day_type` family and, if that subtype has no observed evidence, they fall back to the broader `anchor_holiday_name` history.

In [ ]:
# 9. Holiday selector feature table (historical window for all series)
import numpy as np
from analog_holidays.shared.identify_holidays import build_holiday_selector_features

_SELECTOR_GROUP_COLS = ('unique_id', 'anchor_holiday_name', 'holiday_day_type')
_SELECTOR_DOW_LABELS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
selector_n_clusters = int(N_CLUSTERS) if 'N_CLUSTERS' in globals() else 2
selector_ab_alpha = float(AB_ALPHA) if 'AB_ALPHA' in globals() else 0.10


def _empty_cluster_ab_results():
    return {
        'occurrences_df': pd.DataFrame(),
        'summary_df': pd.DataFrame(),
        'dow_labels': list(_SELECTOR_DOW_LABELS),
    }


def _cluster_38h_profiles_no_plots(
    df_wide: pd.DataFrame,
    match_dates_set: set,
    df_holidays_local: pd.DataFrame,
    hour_cols: list,
    n_clusters: int,
    previously_w_hours: int,
) -> dict:
    cluster_labels = list('CDEFGHIJ')
    eve_hour_cols = hour_cols[-previously_w_hours:]
    hol_hour_cols = hour_cols
    feat_cols = [f'eve_{col_name}' for col_name in eve_hour_cols] + [f'hol_{col_name}' for col_name in hol_hour_cols]

    date_to_name = dict(
        zip(
            pd.to_datetime(df_holidays_local['date']).dt.normalize(),
            df_holidays_local['holiday_name'],
        )
    )
    wide_idx = set(df_wide.index.normalize())
    rows = []
    for target_date in sorted(match_dates_set):
        eve_date = target_date - pd.Timedelta(days=1)
        if eve_date not in wide_idx:
            continue
        row_eve = df_wide.loc[df_wide.index.normalize() == eve_date].squeeze()
        row_hol = df_wide.loc[df_wide.index.normalize() == target_date].squeeze()
        eve_values = row_eve[eve_hour_cols].values.astype(float)
        holiday_values = row_hol[hol_hour_cols].values.astype(float)
        if np.isnan(eve_values).any() or np.isnan(holiday_values).any():
            continue
        rows.append({
            'date': target_date,
            'holiday_name': date_to_name.get(target_date, str(target_date.date())),
            **dict(zip([f'eve_{col_name}' for col_name in eve_hour_cols], eve_values)),
            **dict(zip([f'hol_{col_name}' for col_name in hol_hour_cols], holiday_values)),
        })

    if not rows:
        df_phol = pd.DataFrame(columns=['holiday_name', *feat_cols, 'cluster', 'cluster_type'])
        df_phol.index = pd.DatetimeIndex([], name='date')
        return {
            'df_phol': df_phol,
            'df_days_phol': pd.DataFrame(columns=['type', 'date', 'dow', 'holiday_name']),
            'kmeans_phol': None,
            'centroids_phol': np.empty((0, len(feat_cols))),
            'feat_cols': feat_cols,
        }

    df_phol = pd.DataFrame(rows).set_index('date')
    resolved_n_clusters = min(int(n_clusters), len(df_phol))
    profiles = df_phol[feat_cols].values.astype(float)
    profiles_scaled = identify_holidays_module.StandardScaler().fit_transform(profiles)
    kmeans_phol = identify_holidays_module.KMeans(
        n_clusters=resolved_n_clusters,
        random_state=42,
        n_init=20,
    ).fit(profiles_scaled)
    df_phol['cluster'] = kmeans_phol.labels_
    df_phol['cluster_type'] = [cluster_labels[label] for label in kmeans_phol.labels_]

    centroids_phol = np.array([
        df_phol.loc[df_phol['cluster'] == cluster_id, feat_cols].values.mean(axis=0)
        for cluster_id in range(resolved_n_clusters)
    ])

    df_days_phol = (
        pd.DataFrame(
            [
                {
                    'type': row['cluster_type'],
                    'date': date_value,
                    'dow': date_value.day_name()[:3],
                    'holiday_name': row['holiday_name'],
                }
                for date_value, row in df_phol.iterrows()
            ]
        )
        .sort_values(['type', 'date'])
        .reset_index(drop=True)
    )

    return {
        'df_phol': df_phol,
        'df_days_phol': df_days_phol,
        'kmeans_phol': kmeans_phol,
        'centroids_phol': centroids_phol,
        'feat_cols': feat_cols,
    }


def _build_selector_history_for_unique_id(unique_id: str) -> dict:
    df_raw_full = load_results_data(DEMAND_PATH, unique_id)
    available_dates_full = set(df_raw_full['ds'].dt.normalize())

    df_raw_local = df_raw_full.copy()
    if DATE_END is not None:
        cutoff_ts = pd.Timestamp(DATE_END)
        df_raw_local = df_raw_local[df_raw_local['ds'] < cutoff_ts].copy()
    if df_raw_local.empty:
        raise ValueError(f'No data available for {unique_id} inside the historical selector window.')

    df_wide_local = build_wide_df(df_raw_local, MONTH_TO_SEGMENT, exclude_years=[])
    if df_wide_local.empty:
        raise ValueError(f'No complete daily profiles are available for {unique_id}.')

    hour_cols_local = get_hour_cols(df_wide_local)
    year_min_local = int(df_wide_local.index.year.min())
    year_max_local = int(df_wide_local.index.year.max())
    df_holidays_local = load_holidays_catalog(HOLIDAYS_PATH, year_min_local, year_max_local)
    known_holiday_dates_local = (
        set(pd.to_datetime(df_holidays_local['date']).dt.normalize())
        if EXCLUDE_KNOWN_HOLIDAYS_FROM_BASELINE else set()
    )

    df_dist_local = compute_distances(
        df_wide_local,
        hour_cols_local,
        SEGMENT_LABELS,
        WEEKDAY_NAMES,
        distance_metric=DISTANCE_METRIC,
        reference_exclude_dates=known_holiday_dates_local,
    )
    df_dist_local, df_outliers_local = detect_outliers(
        df_dist_local,
        OUTLIER_PERCENTILE,
        threshold_reference_only=EXCLUDE_KNOWN_HOLIDAYS_FROM_BASELINE,
        promote_dates=known_holiday_dates_local,
        promote_min_group_percentile=KNOWN_HOLIDAY_MIN_GROUP_PERCENTILE,
    )
    df_match_local, _ = compare_outliers_holidays(df_outliers_local, df_holidays_local)
    all_dates_local = set(df_wide_local.index.normalize())
    date_sets_local = get_date_sets(df_match_local, df_holidays_local, all_dates_local)

    match_dates_set_local = date_sets_local['match_dates_set']
    unknown_dates_set_local = date_sets_local['unknown_dates_set']
    outlier_dates_set_local = date_sets_local['outlier_dates_set']
    atypical_dates_local = match_dates_set_local | unknown_dates_set_local

    if atypical_dates_local:
        cluster_results_local = identify_holidays_module.cluster_atypical_profiles(
            df_wide_local,
            match_dates_set_local,
            unknown_dates_set_local,
            outlier_dates_set_local,
            min(selector_n_clusters, len(atypical_dates_local)),
            hour_cols_local,
            df_holidays_local,
        )
        cluster_ab_results_local = identify_holidays_module.classify_cluster_dow_type(
            df_wide_local,
            cluster_results_local['df_atyp'],
            outlier_dates_set_local,
            hour_cols_local,
            selector_ab_alpha,
        )
    else:
        cluster_ab_results_local = _empty_cluster_ab_results()

    cluster_38h_results_local = _cluster_38h_profiles_no_plots(
        df_wide_local,
        match_dates_set_local,
        df_holidays_local,
        hour_cols_local,
        N_CLUSTERS_PHOL,
        PREVIOUSLY_W_HOURS,
    )

    df_selector_history_local = build_holiday_selector_features(
        df_wide=df_wide_local,
        df_holidays=df_holidays_local,
        cluster_ab_results=cluster_ab_results_local,
        df_phol=cluster_38h_results_local['df_phol'],
        holidays_path=HOLIDAYS_PATH,
        unique_id=unique_id,
    )

    print(
        f'[{unique_id}] complete days={len(df_wide_local)} | '
        f'history selector rows={len(df_selector_history_local)} | '
        f'matched holidays={len(match_dates_set_local)}'
    )
    return {
        'unique_id': unique_id,
        'selector_history': df_selector_history_local,
        'available_dates_full': available_dates_full,
    }


selector_series_contexts = {}
selector_history_frames = []
for series_unique_id in AVAILABLE_UNIQUE_IDS:
    selector_context = _build_selector_history_for_unique_id(series_unique_id)
    selector_series_contexts[series_unique_id] = selector_context
    selector_history_frames.append(selector_context['selector_history'])

df_holiday_selector_features_history_all = (
    pd.concat(selector_history_frames, ignore_index=True)
    .sort_values(['unique_id', 'date', 'holiday_name'])
    .reset_index(drop=True)
)
df_holiday_selector_features_history = (
    df_holiday_selector_features_history_all
    .loc[df_holiday_selector_features_history_all['unique_id'] == UNIQUE_ID]
    .copy()
    .reset_index(drop=True)
)

selector_history_counts = (
    df_holiday_selector_features_history_all
    .groupby('unique_id', dropna=False)
    .size()
    .rename('historical_rows')
    .reset_index()
    .sort_values('unique_id')
    .reset_index(drop=True)
)

print(f'Historical selector rows ({UNIQUE_ID}): {len(df_holiday_selector_features_history)}')
print(
    f'Historical selector rows (all series): {len(df_holiday_selector_features_history_all)} '
    f'| series={df_holiday_selector_features_history_all["unique_id"].nunique()}'
)
display(selector_history_counts)
display(df_holiday_selector_features_history)

In [ ]:
# 10. Ex-ante profile priors for new candidates
# Priors are estimated on the historical selector rows of all available series.
from analog_holidays.shared.identify_holidays import (
    build_future_holiday_selector_features,
    build_holiday_selector_priors,
)

selector_features_path = Path('holidays') / 'holiday_selector_features_ercot.csv'
selector_priors_path = Path('holidays') / 'holiday_selector_priors_ercot.csv'

if 'selector_series_contexts' not in globals() or not selector_series_contexts:
    raise ValueError('Run Cell 33 first to build the per-series historical selector contexts.')

df_holiday_selector_priors = build_holiday_selector_priors(
    df_holiday_selector_features_history_all,
    group_cols=_SELECTOR_GROUP_COLS,
)

selector_future_start = (
    pd.Timestamp(DATE_END).normalize()
    if DATE_END is not None
    else pd.to_datetime(df_holiday_selector_features_history_all['date']).max().normalize() + pd.Timedelta(days=1)
)
selector_future_end = pd.Timestamp(SELECTOR_FUTURE_END).normalize() if SELECTOR_FUTURE_END is not None else None
future_year_end = (
    int(selector_future_end.year)
    if selector_future_end is not None
    else int(pd.to_datetime(df_holiday_selector_features_history_all['date']).max().year)
)
df_holidays_selector_source = load_holidays_catalog(
    HOLIDAYS_PATH,
    year_min,
    future_year_end,
)

future_frames = []
for series_unique_id, series_context in selector_series_contexts.items():
    df_future_local = build_future_holiday_selector_features(
        df_holidays=df_holidays_selector_source,
        df_priors=df_holiday_selector_priors,
        available_dates=series_context['available_dates_full'],
        holidays_path=HOLIDAYS_PATH,
        group_cols=_SELECTOR_GROUP_COLS,
        start_date=selector_future_start,
        end_date=selector_future_end,
        unique_id=series_unique_id,
    )
    future_frames.append(df_future_local)
    print(f'[{series_unique_id}] future ex-ante rows={len(df_future_local)}')

df_holiday_selector_features_future_all = (
    pd.concat(future_frames, ignore_index=True)
    .sort_values(['unique_id', 'date', 'holiday_name'])
    .reset_index(drop=True)
    if future_frames else pd.DataFrame(columns=df_holiday_selector_features_history_all.columns)
)
df_holiday_selector_features_future = (
    df_holiday_selector_features_future_all
    .loc[df_holiday_selector_features_future_all['unique_id'] == UNIQUE_ID]
    .copy()
    .reset_index(drop=True)
)

df_holiday_selector_features = (
    pd.concat(
        [df_holiday_selector_features_history_all, df_holiday_selector_features_future_all],
        ignore_index=True,
    )
    .sort_values(['unique_id', 'date', 'holiday_name'])
    .reset_index(drop=True)
)

df_holiday_selector_features_export = df_holiday_selector_features.copy()
df_holiday_selector_features_export['date'] = (
    pd.to_datetime(df_holiday_selector_features_export['date'])
    .dt.strftime('%Y-%m-%d')
)
df_holiday_selector_features_export.to_csv(selector_features_path, index=False)

df_holiday_selector_priors_export = df_holiday_selector_priors.copy()
df_holiday_selector_priors_export.to_csv(selector_priors_path, index=False)

selector_future_counts = (
    df_holiday_selector_features_future_all
    .groupby('unique_id', dropna=False)
    .size()
    .rename('future_rows')
    .reset_index()
    .sort_values('unique_id')
    .reset_index(drop=True)
    if not df_holiday_selector_features_future_all.empty else pd.DataFrame(columns=['unique_id', 'future_rows'])
)

print(f'Historical selector rows ({UNIQUE_ID}): {len(df_holiday_selector_features_history)}')
print(f'Future ex-ante rows ({UNIQUE_ID}): {len(df_holiday_selector_features_future)}')
print(f'Total selector rows exported: {len(df_holiday_selector_features_export)}')
print(
    f'Series exported: {df_holiday_selector_features_export["unique_id"].nunique()} '
    f'| selector_future_start={selector_future_start.date()} '
    f'| selector_future_end={selector_future_end.date() if selector_future_end is not None else "source max"}'
)
print(f'Selector CSV saved to: {selector_features_path.resolve()}')
print(f'Priors CSV saved to: {selector_priors_path.resolve()}')
display(selector_future_counts)
display(
    df_holiday_selector_priors
    .loc[df_holiday_selector_priors['unique_id'] == UNIQUE_ID]
    .reset_index(drop=True)
)
display(df_holiday_selector_features_future)

## 11. Analog-Space Clusters (`F`, `G`, `H`, ...)

Use `CLUSTERING_CRITERIUM` to choose how daily selector rows are grouped into stable analog-space labels.

The public criterion catalog exposed in this notebook currently includes `shape_pearson_CDE_map_FGH`, `seasonal_heat_cold`, `seasonal_winter_sprint_fall`, and `best_matching_weekday`. The binary seasonal option collapses Spring/Summer into heat and Autumn/Winter into cold.

Historical rows determine the analog-space mapping. Future holiday rows in the 2025/2026 test horizon receive their profile labels ex-ante from `holiday_selector_priors.csv`, so the exported selector can be filled without using future observations.

The selector export `holidays/holiday_selector_features.csv` is the single source of truth for daily `analog_cluster` labels. No hourly `*_cluster` columns are added to the demand source anymore.

In [ ]:
from analog_holidays.shared.identify_holidays import assign_holiday_selector_analog_clusters

analog_cluster_results = assign_holiday_selector_analog_clusters(
    df_selector=df_holiday_selector_features,
    df_priors=df_holiday_selector_priors,
    criterion=CLUSTERING_CRITERIUM,
    group_cols=_SELECTOR_GROUP_COLS,
    cluster_labels=('F', 'G', 'H'),
)
df_holiday_selector_analog_clusters = analog_cluster_results['df_selector_clusters']
df_analog_cluster_catalog = analog_cluster_results['analog_cluster_catalog']
analog_criterion_prior_col = analog_cluster_results.get('analog_criterion_prior_col')

drop_columns = [
    'analog_criterion',
    'analog_criterion_value',
]
if analog_criterion_prior_col is not None:
    drop_columns.append(analog_criterion_prior_col)

df_holiday_selector_features = (
    df_holiday_selector_analog_clusters
    .drop(columns=drop_columns, errors='ignore')
    .assign(analog_cluster_criterion=CLUSTERING_CRITERIUM)
    .sort_values(['unique_id', 'date', 'holiday_name'])
    .reset_index(drop=True)
)
df_holiday_selector_features_current = (
    df_holiday_selector_features
    .loc[df_holiday_selector_features['unique_id'] == UNIQUE_ID]
    .copy()
    .reset_index(drop=True)
)

_analog_cluster_by_date = (
    df_holiday_selector_features_current[['date', 'analog_cluster']]
    .dropna(subset=['analog_cluster'])
    .drop_duplicates(subset=['date'])
    .copy()
)
_analog_cluster_by_date['date'] = pd.to_datetime(_analog_cluster_by_date['date']).dt.normalize()

df_holidays_display = (
    df_holidays_display
    .drop(columns=['analog_cluster'], errors='ignore')
    .merge(_analog_cluster_by_date, on='date', how='left')
)

df_holiday_selector_features_export = df_holiday_selector_features.copy()
if 'date' in df_holiday_selector_features_export.columns:
    df_holiday_selector_features_export['date'] = (
        pd.to_datetime(df_holiday_selector_features_export['date'])
        .dt.strftime('%Y-%m-%d')
    )
df_holiday_selector_features_export.to_csv(selector_features_path, index=False)

df_analog_cluster_summary = (
    df_holiday_selector_features
    .dropna(subset=['analog_cluster'])
    .groupby(['unique_id', 'analog_cluster'], dropna=False)
    .size()
    .rename('n_rows')
    .reset_index()
    .sort_values(['unique_id', 'analog_cluster'])
    .reset_index(drop=True)
)

print(f'Analog-cluster criterion: {CLUSTERING_CRITERIUM}')
print(f'Catalog rows: {len(df_analog_cluster_catalog)}')
print(
    f'Analog labels exported: {tuple(df_analog_cluster_catalog["analog_cluster"].astype(str).tolist())}'
)
print(
    f'Selector CSV saved to: {selector_features_path.resolve()} '
    f'| series exported={df_holiday_selector_features_export["unique_id"].nunique()}'
)
print('df_holidays_display updated with analog_cluster for the current detail-view series. Re-run the figure cells in sections 7 and 8 to see labels like [F].')
display(df_analog_cluster_catalog)
display(df_analog_cluster_summary)
display(
    df_holiday_selector_features_current[
        [
            'unique_id',
            'holiday_name',
            'anchor_holiday_name',
            'date',
            'holiday_day_type',
            'event_profile_cluster',
            'analog_cluster_criterion',
            'analog_cluster',
        ]
    ].sort_values('date').reset_index(drop=True)
)

In [ ]:
analog_cluster_labels = tuple(
    df_analog_cluster_catalog['analog_cluster']
    .dropna()
    .astype(str)
    .tolist()
)
if not analog_cluster_labels:
    analog_cluster_labels = ('F', 'G', 'H')

analog_cluster_fgh_results = run_analog_cluster_38h_analysis(
    df_phol=df_phol,
    df_holiday_selector_features=df_holiday_selector_features,
    cluster_colors=CLUSTER_COLORS,
    unique_id=UNIQUE_ID,
    feat_cols=cluster_38h_results['feat_cols'],
    previously_w_hours=PREVIOUSLY_W_HOURS,
    cluster_labels=analog_cluster_labels,
    selection_criterion=CLUSTERING_CRITERIUM,
)

df_phol_fgh = analog_cluster_fgh_results['df_phol_fgh']
df_days_fgh = analog_cluster_fgh_results['df_days_fgh']
centroids_fgh = analog_cluster_fgh_results['centroids_fgh']
feat_cols = analog_cluster_fgh_results['feat_cols']

## 12. Final summary

In [ ]:
print_summary(
    UNIQUE_ID, df_wide, n_total, n_match, n_unknown,
    df_missed, df_outliers_cmp, OUTLIER_PERCENTILE, WEEKDAY_NAMES,
)